In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
new_df = pd.read_csv("../../data/processed/final_EDA_df.csv")
new_df.head()

,고객ID,패션뉴스구독여부,멤버십상태,뉴스레터수신빈도,연령,구매일,가격,상품그룹,연령대,거래날짜,이전구매일,구매간격(일),API ratio,전체고객이탈여부,신규고객이탈
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2018-12-27,0.044051,Garment Upper body,40대,2018-12-27,0,0.0,68.666667,0,0
1,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2018-12-27,0.035576,Garment Upper body,40대,2018-12-27,2018-12-27 00:00:00,0.0,68.666667,0,0
2,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2018-12-27,0.030492,Garment Upper body,40대,2018-12-27,2018-12-27 00:00:00,0.0,68.666667,0,0
3,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2019-05-02,0.010153,Garment Full body,40대,2019-05-02,2018-12-27 00:00:00,126.0,68.666667,0,0
4,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2019-05-25,0.050831,Garment Upper body,40대,2019-05-25,2019-05-02 00:00:00,23.0,68.666667,0,0


In [6]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE


# target = "신규고객이탈여부"
y = new_df['신규고객이탈여부']
X = new_df.drop(columns=["고객ID","신규고객이탈여부"]) # 가입일과 이탈일은 모델링에 사용하지 않음

# drop_cols = ["고객ID"]
# X = X.drop(columns=[c for c in drop_cols if c in X.columns], errors="ignore") # 제거할 열이 존재하지 않을 경우 오류 방지


print(f'만들어진 샘플 비율 : {np.bincount(y)}') # 균형이 맞지 않음

smote = SMOTE(random_state =42)
X_resample, y_resample = smote.fit_resample(X,y)

print(f'resample 후 샘플 비율 : {np.bincount(y_resample)}')  # 균형이 잡힘



KeyError: '신규고객이탈여부'

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

# 오버샘플링 전 데이터 활용

X_train, X_test, y_train, y_test = train_test_split(X_resample, y_resample, test_size=0.2, random_state=42)

rf_clf = XGBClassifier(random_state=0)
rf_clf.fit(X_train, y_train)

y_pred = rf_clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    245287
           1       1.00      1.00      1.00    245147

    accuracy                           1.00    490434
   macro avg       1.00      1.00      1.00    490434
weighted avg       1.00      1.00      1.00    490434



In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import numpy as np

best = (None, 0)

num_cols = X_train.select_dtypes(include=[np.number]).columns

for c in num_cols:
    stump = DecisionTreeClassifier(max_depth=1, random_state=42)
    stump.fit(X_train[[c]], y_train)
    acc = accuracy_score(y_test, stump.predict(X_test[[c]]))
    if acc > best[1]:
        best = (c, acc)

print("한 개 피처만으로 나온 최고 정확도:", best)

한 개 피처만으로 나온 최고 정확도: ('신규고객_총구매횟수_F', 1.0)


In [ ]:
from xgboost import XGBClassifier


# xgb_clf = XGBClassifier( # 하이퍼파라미터 튜닝
#     # n_estimators=100,       # 트리의 개수       
#     # max_depth=6,            # 트리의 최대 깊이
#     # learning_rate=0.1, 
#     # random_state=0
# )
# xgb_clf.fit(X_train, y_train)


xgb_clf = XGBClassifier(random_state=0)
xgb_clf.fit(X_train, y_train)

y_pred_test = xgb_clf.predict(X_test)

print(classification_report(y_test, y_pred_test))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    245287
           1       1.00      1.00      1.00    245147

    accuracy                           1.00    490434
   macro avg       1.00      1.00      1.00    490434
weighted avg       1.00      1.00      1.00    490434



In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_train = xgb_clf.predict(X_train)
y_pred_test = xgb_clf.predict(X_test)

print(accuracy_score(y_train, y_pred_train))
print(accuracy_score(y_test, y_pred_test))

print(classification_report(y_test, y_pred_test))



1.0
1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    245287
           1       1.00      1.00      1.00    245147

    accuracy                           1.00    490434
   macro avg       1.00      1.00      1.00    490434
weighted avg       1.00      1.00      1.00    490434

